# Real signed PDI and post-quantum boot

These cells use **recorded hardware evidence** by default. They do not program a board.
Use the configured Python (SCU35 PMC) kernel. Fresh hardware commands are shown in Markdown and must be invoked explicitly after configuring your exact SCU35 and recovery image.
Current PL timing uses AXI Timer; native PMC timing uses `mcycle`. JTAG/UART are outside operation timing.

`python lab.py boot` packages your new real design and uses fresh signing state. The following cells verify the retained H15 PDI signatures without private keys. Positive and tampered hardware captures remain separate from offline verification. No owner-pinned authentication policy was established; unsigned images were also accepted.

In [1]:
from pathlib import Path
import hashlib, json, sys, time
import pandas as pd
# Works from the notebook directory, lab root, or parent repository.
candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for base in candidates for p in (base, base / "scu35") if (p / "lab.py").is_file()), None)
assert ROOT is not None, "Launch Jupyter inside this checkout or its scu35 directory"
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "host"))
from compare import parse_capture
from lab import config
capture = parse_capture(ROOT / "evidence/timer/uart.txt", "7d1bf77bc454dd1f")
rows = pd.DataFrame(capture["rows"])
print("Recorded board capture: 31 complete rows; current Python:", sys.version.split()[0])


Recorded board capture: 31 complete rows; current Python: 3.11.9


In [2]:
sys.path.insert(0, str(ROOT / "scripts"))
from pdi import verify_signed
signed = ROOT / "reference/boot/signed.pdi"
result, _ = verify_signed(signed.read_bytes(), ROOT / "evidence/boot/bundle")
assert all(result[k] for k in ["spk_signature_valid", "hash_block_signature_valid", "boot_header_hash_valid"])
display(result)
print("PDI SHA-256:", hashlib.sha256(signed.read_bytes()).hexdigest())

{'header': {'key_source': 0,
  'plm_offset': 20400,
  'pl_load': 4060086272,
  'pl_bytes': 1327536,
  'pl_encoded': 1332720,
  'plm_bytes': 26048,
  'plm_encoded': 26144,
  'attributes': 786432},
 'ppk_sha256': '37c3a77aa8d423c528bdc140a689a0f99bd1e10f543cea788d90aff923cfa943',
 'spk_sha256': 'b17adea47869aa3629fb00ada6c4ae9cf323282c3d857110b6cc80e300e98825',
 'spk_signature_valid': True,
 'hash_block_signature_valid': True,
 'boot_header_hash_valid': True,
 'signature_bytes': 9620,
 'signature_offset': 10768,
 'hash_block_offset': 10624,
 'plm_sha256': 'bdafedd897fe86e0f078cf76339baaa0f18826292efc04736c5b08ec9f464498',
 'pl_sha256': '4bef6bc8070939095bd8e563caa1c0eb9d76ffb47758d8130a079f50192fb3a9',
 'plm_bytes': 26048,
 'pl_bytes': 1327536,
 'chain_chunks': 167}

PDI SHA-256: 8ab7ac3f967cdab595930f955eade363ba72d5b46930c51394e7a1c72650da54


## Fresh hardware: explicit step

Configure `config.json` first. Run `python lab.py hardware --pdi /path/to/build/demo.pdi` from the lab directory for the full runtime suite, or the native/boot command above for that path. Save logs, verify recovery and compare results before discussing speed. See [audit checkpoints](../docs/audit.md).